# PaySim Feature Engineering

Dataset: PaySim Fraud Detection

| Step | Note |
|------|------|
| Memory Optimization | Downcasts numeric data types to reduce memory usage |
| Data Loading | Loads raw PaySim transaction data and sorts by time |
| Baseline Cleaning | Encodes categorical transaction type and keeps raw baseline features |
| Domain-Specific Features | Adds transaction consistency, balance anomaly, and temporal behavior features |
| Output | Saves baseline and engineered datasets to Parquet files |

In [1]:
import pandas as pd
import numpy as np
import gc
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

DATA_PATH = 'data_raw/PaySim/PS_20174392719_1491204439457_log.csv'

# 1. MEMORY OPTIMIZATION
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if str(col_type) in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min, c_max = df[col].min(), df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Mem decreased to {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# 2. LOAD DATA
print("Loading data...")
df = pd.read_csv(DATA_PATH)

# Sort by Time (Chronological Splitting Requirement)
df = df.sort_values('step').reset_index(drop=True)

print(df.shape)
df.head()

# 3. BASELINE CLEANING
print("Applying baseline cleaning...")

# Encode transaction type
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'].astype(str))

# Drop high-cardinality identifier columns for ML baseline
df_baseline = df.drop(columns=['nameOrig', 'nameDest'], errors='ignore').copy()

# Optimize and save baseline parquet
df_baseline = reduce_mem_usage(df_baseline)
df_baseline.to_parquet('data/paysim_baseline.parquet', index=False)

print("Baseline dataset saved: data/paysim_baseline.parquet")

# 4. DOMAIN-SPECIFIC FEATURE ENGINEERING (PaySim-specific)
print("Adding Domain-Specific Features...")

df_eng = df_baseline.copy()

# A. TRANSACTION CONSISTENCY FEATURES
# In genuine transfers, balances should move consistently with amount.
df_eng['orig_error'] = df_eng['oldbalanceOrg'] - df_eng['amount'] - df_eng['newbalanceOrig']
df_eng['dest_error'] = df_eng['oldbalanceDest'] + df_eng['amount'] - df_eng['newbalanceDest']

# B. BALANCE CHANGE FEATURES
df_eng['orig_balance_diff'] = df_eng['oldbalanceOrg'] - df_eng['newbalanceOrig']
df_eng['dest_balance_diff'] = df_eng['newbalanceDest'] - df_eng['oldbalanceDest']

# C. RATIO / RELATIVE FEATURES
df_eng['amt_to_oldbalance_org'] = df_eng['amount'] / (df_eng['oldbalanceOrg'] + 1)
df_eng['amt_to_oldbalance_dest'] = df_eng['amount'] / (df_eng['oldbalanceDest'] + 1)

# D. ZERO BALANCE FLAGS
df_eng['orig_zero_before'] = (df_eng['oldbalanceOrg'] == 0).astype(np.int8)
df_eng['orig_zero_after'] = (df_eng['newbalanceOrig'] == 0).astype(np.int8)
df_eng['dest_zero_before'] = (df_eng['oldbalanceDest'] == 0).astype(np.int8)
df_eng['dest_zero_after'] = (df_eng['newbalanceDest'] == 0).astype(np.int8)

# E. TEMPORAL FEATURES
df_eng['hour'] = (df_eng['step'] % 24).astype(np.int8)
df_eng['day'] = (df_eng['step'] // 24).astype(np.int16)

# F. LARGE TRANSACTION FLAG
amt_95 = df_eng['amount'].quantile(0.95)
df_eng['is_large_transaction'] = (df_eng['amount'] > amt_95).astype(np.int8)

# G. FRAUD-PRONE TRANSACTION TYPE INTERACTION
# CASH_OUT and TRANSFER are usually most relevant in PaySim
df_eng['type_amount_interaction'] = df_eng['type'] * df_eng['amount']

# Fill any numerical anomalies
df_eng = df_eng.replace([np.inf, -np.inf], np.nan).fillna(-999)

# Optimize and save engineered parquet
df_eng = reduce_mem_usage(df_eng)
df_eng.to_parquet('data/paysim_feature.parquet', index=False)

print("Process Complete. Produced: paysim_baseline.parquet and paysim_feature.parquet")

Loading data...
(6362620, 11)
Applying baseline cleaning...
Mem decreased to 151.70 Mb (65.3% reduction)
Baseline dataset saved: data/paysim_baseline.parquet
Adding Domain-Specific Features...
Mem decreased to 364.07 Mb (1.6% reduction)
Process Complete. Produced: paysim_baseline.parquet and paysim_feature.parquet
